# GISPR Module 6 — Raster Operations in Python and R

**Course:** GIS Spatial Analysis with Python and R (GISPR)  
**Module:** 6 — Raster Operations: Map Algebra, Warping, Zonal Statistics, and Multidimensional Arrays  
**Builds on:** Module 5 (raster data model, `inspect_raster()` checklist, array indexing, NoData handling)

---

### What you'll accomplish this module

By the end of this notebook you'll be able to:

| # | Learning outcome | Where |
|---|---|---|
| 1 | Execute raster geoprocessing with ESRI tools — raster algebra, reclassification, and format conversion via `arcpy.sa` | Section 2 |
| 2 | Perform cell-level map algebra and reclassification with NumPy/rasterio and terra | Section 2 |
| 3 | Warp rasters — reproject, resample, and clip to a study area — with `rasterio.warp` and `terra` | Section 3 |
| 4 | Run neighborhood (focal) operations and derive slope and aspect from a DEM | Section 4 |
| 5 | Calculate zonal statistics over vector polygon zones — via the ESRI path and the open-source path | Section 5 |
| 6 | Convert between data models: rasterize vector zones, vectorize class rasters | Section 6 |
| 7 | Analyze multidimensional time-series datasets (NetCDF) with `xarray` and `stars` | Section 7 |

### Kernel reminder
- 🔵 **`[R]`** cells — switch kernel to **R** before running
- 🟢 **`[Python]`** cells — switch kernel to **Python 3** (or your cloned ArcGIS Pro env) before running
- 📋 **`[Terminal]`** cells — paste the command into your terminal / Anaconda Prompt, **not** in a notebook cell
- 🏠 **`LOCAL ONLY`** cells — require ArcGIS Pro on your own machine; they will not run on JupyterHub

> **The framing problem for this module:**  
> The regional salmon recovery board needs a habitat prioritization table: **for every watershed in the study area** — mean elevation, mean slope, percent forest cover, and whether the watershed contains low-elevation floodplain (below 100 m).  
> The DEM arrives in geographic coordinates (EPSG:4326) at a different resolution than the land cover grid (EPSG:32610). Doing this in Pro means Project Raster → Slope → Reclassify → Zonal Statistics as Table → joins — for every update. By the end of this notebook, the whole thing is one function call.

---
## Section 1 — Setup: The Synthetic Study Area

As in Module 5, we **generate** our rasters instead of downloading them, so every cell runs identically on JupyterHub, your laptop, or a colleague's machine. The factory below creates a small Pacific-Northwest-style study area:

| File | Contents | CRS | dtype | Why it exists |
|---|---|---|---|---|
| `data/mod6/dem_32610.tif` | Elevation (m), ridge–valley terrain | EPSG:32610 | float32 | Map algebra, terrain, zonal stats |
| `data/mod6/dem_4326.tif` | The *same* terrain, coarse, geographic | EPSG:4326 | float32 | The misaligned delivery — motivates warping |
| `data/mod6/landcover_32610.tif` | NLCD-style classes (11, 21, 41, 42, 52, 71, 90) | EPSG:32610 | uint8 | Categorical operations, zonal composition |
| `data/mod6/watersheds.gpkg` | 5 watershed polygons with `ws_id`, `ws_name` | EPSG:32610 | — | The zones for zonal statistics |
| `data/mod6/ndvi_monthly.nc` | 12 monthly NDVI grids (one year) | EPSG:32610 | float32 | Multidimensional arrays with xarray/stars |

Run the factory once, then treat the files as if they arrived from a data provider.

In [ ]:
# [Python] Synthetic data factory — run once; creates data/mod6/
# Everything is generated locally: no downloads, no credentials, no license.

import numpy as np
import rasterio
from rasterio.transform import from_origin
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio import features
import geopandas as gpd
from shapely.geometry import shape
import xarray as xr
from pathlib import Path

rng = np.random.default_rng(42)
DATA = Path("data/mod6")
DATA.mkdir(parents=True, exist_ok=True)

# ── Grid definition: 200 x 200 cells at 30 m in EPSG:32610 (course standard) ──
ROWS, COLS, CELL = 200, 200, 30.0
X0, Y0 = 550_000.0, 5_280_000.0            # NW corner, UTM 10N — near Puget Sound
transform = from_origin(X0, Y0, CELL, CELL)

# ── 1. DEM: two ridges and a river valley, in metres ─────────────────────────
yy, xx = np.mgrid[0:ROWS, 0:COLS]
dem = (
    650 * np.exp(-((xx - 45) ** 2 + (yy - 60) ** 2) / (2 * 55 ** 2))     # west ridge
    + 900 * np.exp(-((xx - 150) ** 2 + (yy - 130) ** 2) / (2 * 45 ** 2)) # east ridge
    + 40                                                                  # valley floor
    + rng.normal(0, 4, (ROWS, COLS))                                      # micro-relief
).astype("float32")
dem[dem < 0] = 0

profile = dict(driver="GTiff", height=ROWS, width=COLS, count=1,
               dtype="float32", crs="EPSG:32610", transform=transform,
               nodata=-9999.0)
with rasterio.open(DATA / "dem_32610.tif", "w", **profile) as dst:
    dst.write(dem, 1)

# ── 2. The 'delivered' DEM: same terrain, reprojected to EPSG:4326, coarse ───
with rasterio.open(DATA / "dem_32610.tif") as src:
    t4326, w4326, h4326 = calculate_default_transform(
        src.crs, "EPSG:4326", src.width, src.height, *src.bounds,
        resolution=0.0006)                       # ~60 m — deliberately coarser
    p4326 = src.profile | dict(crs="EPSG:4326", transform=t4326,
                               width=w4326, height=h4326)
    with rasterio.open(DATA / "dem_4326.tif", "w", **p4326) as dst:
        reproject(rasterio.band(src, 1), rasterio.band(dst, 1),
                  resampling=Resampling.bilinear)

# ── 3. Land cover: elevation- and noise-driven NLCD-style classes ────────────
lc = np.full((ROWS, COLS), 52, dtype="uint8")            # shrub by default
lc[dem < 55] = 11                                        # water in the valley floor
lc[(dem >= 55) & (dem < 90)] = 90                        # wetlands fringe
lc[(dem >= 90) & (dem < 150) & (xx < 100)] = 21          # development, west valley
lc[(dem >= 150) & (dem < 450)] = 41                      # deciduous forest
lc[dem >= 450] = 42                                      # evergreen forest
grass = rng.random((ROWS, COLS)) < 0.06
lc[grass & (dem >= 90) & (dem < 450)] = 71               # grassland patches

lc_profile = profile | dict(dtype="uint8", nodata=0)
with rasterio.open(DATA / "landcover_32610.tif", "w", **lc_profile) as dst:
    dst.write(lc, 1)

# ── 4. Watersheds: nearest-seed (Voronoi) basins, vectorized to polygons ─────
seeds = np.array([[40, 40], [50, 155], [120, 30], [130, 105], [175, 170]])  # row, col
dists = np.stack([np.hypot(yy - r, xx - c) for r, c in seeds])
ws_grid = (dists.argmin(axis=0) + 1).astype("uint8")     # 1..5

names = {1: "Cedar Creek", 2: "Boulder Fork", 3: "Silver Run",
         4: "Marsh Slough", 5: "Granite Basin"}
recs = [{"ws_id": int(v), "ws_name": names[int(v)], "geometry": shape(g)}
        for g, v in features.shapes(ws_grid, transform=transform)]
watersheds = (gpd.GeoDataFrame(recs, crs="EPSG:32610")
                .dissolve(by="ws_id", as_index=False, aggfunc="first"))
watersheds.to_file(DATA / "watersheds.gpkg", layer="watersheds", driver="GPKG")

# ── 5. Monthly NDVI: 12 time steps with a seasonal green-up curve ────────────
months = np.arange(1, 13)
season = 0.25 + 0.35 * np.sin((months - 4) / 12 * 2 * np.pi)   # peaks mid-summer
base = 0.15 + 0.5 * (lc >= 41) * (lc <= 71)                    # vegetated classes greener
ndvi = np.stack([np.clip(base * (0.6 + s) + rng.normal(0, 0.03, (ROWS, COLS)), -0.1, 0.95)
                 for s in season]).astype("float32")
xcoords = X0 + (np.arange(COLS) + 0.5) * CELL
ycoords = Y0 - (np.arange(ROWS) + 0.5) * CELL
ds = xr.Dataset(
    {"ndvi": (("time", "y", "x"), ndvi)},
    coords={"time": [f"2025-{m:02d}-15" for m in months],
            "y": ycoords, "x": xcoords},
    attrs={"crs": "EPSG:32610", "description": "Synthetic monthly NDVI, GISPR Module 6"})
ds = ds.assign_coords(time=np.array([f"2025-{m:02d}-15" for m in months],
                                    dtype="datetime64[ns]"))
ds.to_netcdf(DATA / "ndvi_monthly.nc")

print("Created:")
for f in sorted(DATA.iterdir()):
    print(f"  {f}  ({f.stat().st_size/1024:.0f} KB)")

### 1a — Re-establish the Module 5 inspection habit

Every raster operation in this module **starts** with the first-inspection checklist. `inspect_raster()` below is the same function you built in Module 5 — copy it into any project. If the CRS, resolution, or NoData surprises you here, it will corrupt everything downstream.

In [ ]:
# [Python] inspect_raster() — carried forward from Module 5
import rasterio
import numpy as np

def inspect_raster(path):
    """Module 5 first-inspection checklist: run before ANY raster operation."""
    with rasterio.open(path) as src:
        arr = src.read(1, masked=True)
        print(f"── {path} " + "─" * max(0, 60 - len(str(path))))
        print(f"  CRS        : {src.crs}")
        print(f"  Extent     : {tuple(round(v, 1) for v in src.bounds)}")
        print(f"  Resolution : {src.res}")
        print(f"  Size       : {src.width} x {src.height} px, {src.count} band(s)")
        print(f"  dtype      : {src.dtypes[0]}   NoData: {src.nodata}")
        print(f"  Value range: {arr.min():.2f} to {arr.max():.2f}  (mean {arr.mean():.2f})")
    return None

inspect_raster("data/mod6/dem_32610.tif")
inspect_raster("data/mod6/dem_4326.tif")     # ⚠ different CRS AND resolution
inspect_raster("data/mod6/landcover_32610.tif")

In [ ]:
# [R] The same inspection with terra — run after switching to the R kernel
# terra's SpatRaster prints most of the checklist in one call

library(terra)

dem <- rast("data/mod6/dem_32610.tif")
dem                     # CRS, extent, resolution, dtype in one printout
global(dem, c("min", "max", "mean"), na.rm = TRUE)

lc <- rast("data/mod6/landcover_32610.tif")
freq(lc)                # cell counts per land cover class — categorical sanity check

> 🔧 **Try it yourself:** Run `inspect_raster()` on `dem_4326.tif` and write down the two properties that differ from `dem_32610.tif`. Those two differences are exactly what Section 3 (warping) will fix.

---
## Section 2 — Map Algebra: Cell-Level Calculations and Reclassification

Map algebra is arithmetic where every operand is a grid. Dana Tomlin's classic taxonomy organizes raster operations by *how many cells contribute to each output cell* — and it maps directly onto this module:

| Scope | Each output cell depends on… | Examples | Section |
|---|---|---|---|
| **Local** | the same cell in the input(s) | `dem * 3.28`, reclassify, raster A + raster B | **2 (here)** |
| **Focal** | a neighborhood around the cell | focal mean, slope, aspect, hillshade | 4 |
| **Zonal** | all cells sharing a zone | mean elevation per watershed | 5 |
| **Global** | every cell in the raster | min / max / mean / std | Module 5 ✔ |

Because Module 5 taught you that *rasters are arrays*, local map algebra costs you nothing new: it's NumPy arithmetic in Python and plain operators in terra. Following course convention, the ESRI path comes first.

In [ ]:
# [Python / ArcPy] ESRI path — raster algebra, reclassification, format conversion
# Reference cell: requires ArcGIS Pro + Spatial Analyst. Falls back gracefully on JupyterHub.

try:
    import arcpy
    from arcpy.sa import Raster, RemapRange, Reclassify

    arcpy.CheckOutExtension("Spatial")           # ← forgetting this = licensing error
    arcpy.env.workspace = "data/mod6"
    arcpy.env.overwriteOutput = True

    # 1. Raster algebra — Raster() objects overload +, -, *, /, comparisons
    dem_ft  = Raster("dem_32610.tif") * 3.28084          # metres → feet
    lowland = Raster("dem_32610.tif") < 100              # boolean floodplain mask

    # 2. Reclassify — same tool as the Pro toolbox, scripted
    remap = RemapRange([[0, 100, 1], [100, 300, 2], [300, 600, 3], [600, 2000, 4]])
    elev_class = Reclassify("dem_32610.tif", "VALUE", remap)
    elev_class.save("elev_class_arcpy.tif")

    # 3. Format conversion
    arcpy.management.CopyRaster("dem_32610.tif", "dem_32610.crf")   # cloud raster format

    arcpy.CheckInExtension("Spatial")
    print("ArcPy raster geoprocessing complete.")

except ModuleNotFoundError:
    print("arcpy not available in this environment (expected on JupyterHub).")
    print("This cell is reference material — run it in your ArcGIS Pro Python env.")

In [ ]:
# [Python] Open-source path — map algebra is NumPy arithmetic
import rasterio
import numpy as np
import matplotlib.pyplot as plt

with rasterio.open("data/mod6/dem_32610.tif") as src:
    dem = src.read(1, masked=True)
    dem_profile = src.profile

# ── Local algebra: every expression operates cell-by-cell ────────────────────
dem_ft   = dem * 3.28084                     # unit conversion
lowland  = dem < 100                         # boolean mask: candidate floodplain
relief   = dem - dem.min()                   # relative relief above valley floor

print(f"Lowland (<100 m) cells: {lowland.sum()} "
      f"({100 * lowland.sum() / dem.count():.1f}% of study area)")

# ── Reclassify: np.digitize assigns each cell to a bin ───────────────────────
bins = [100, 300, 600]                       # class breaks in metres
elev_class = np.digitize(dem.filled(np.nan), bins).astype("uint8") + 1
elev_class[dem.mask] = 0                     # 0 = NoData class
# classes: 1 = <100 (valley)  2 = 100–300 (foothill)  3 = 300–600 (montane)  4 = >600 (ridge)

with rasterio.open("data/mod6/elev_class.tif", "w",
                   **(dem_profile | {"dtype": "uint8", "nodata": 0})) as dst:
    dst.write(elev_class, 1)

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ax[0].imshow(dem, cmap="terrain"); ax[0].set_title("DEM (m)")
im = ax[1].imshow(elev_class, cmap="YlGnBu", vmin=0, vmax=4)
ax[1].set_title("Reclassified: 4 elevation classes")
plt.colorbar(im, ax=ax[1], shrink=0.8, ticks=[1, 2, 3, 4])
plt.tight_layout(); plt.show()

vals, counts = np.unique(elev_class[elev_class > 0], return_counts=True)
for v, c in zip(vals, counts):
    print(f"  class {v}: {c:>6} cells  ({100*c/counts.sum():.1f}%)")

In [ ]:
# [R] Map algebra and classify() with terra
library(terra)

dem <- rast("data/mod6/dem_32610.tif")

# ── Local algebra: operators apply cell-by-cell, exactly like NumPy ──────────
dem_ft  <- dem * 3.28084
lowland <- dem < 100
relief  <- dem - global(dem, "min", na.rm = TRUE)[[1]]

cat("Lowland cells:", global(lowland, "sum", na.rm = TRUE)[[1]], "\n")

# ── Reclassify: classify() takes a from-to-becomes matrix ────────────────────
rcl <- matrix(c(  0,  100, 1,     # valley
                100,  300, 2,     # foothill
                300,  600, 3,     # montane
                600, 2000, 4),    # ridge
              ncol = 3, byrow = TRUE)
elev_class <- classify(dem, rcl)
freq(elev_class)                                 # cells per class

writeRaster(elev_class, "data/mod6/elev_class_r.tif",
            datatype = "INT1U", overwrite = TRUE)

plot(elev_class, main = "Elevation classes (terra)")

# ESRI equivalents: Raster Calculator (algebra) · Reclassify (classify)

> 🔧 **Try it yourself:** Add a fifth class that splits the valley at 55 m — everything below 55 m is the modeled water surface. In Python change `bins`; in R add a row to `rcl`. Re-run the class counts. How many cells moved?

---
## Section 3 — Warping: Reproject, Resample, and Clip

Your Section 1 inspection found the problem: `dem_4326.tif` is in a **different CRS and resolution** than the land cover grid. Rasters can only be combined when they share the same grid — same CRS, same resolution, same alignment. **Warping** fixes all three at once by resampling cell values onto a new grid.

Two rules professionals live by:

1. **Warp once, early, and document it.** Every resample degrades the data slightly — never chain reprojections.
2. **Match the resampling method to the data type:**

| Data | Method | Why |
|---|---|---|
| Continuous (DEM, NDVI, temperature) | `bilinear` or `cubic` | Interpolation between values is meaningful |
| Categorical (land cover, zones, class grids) | `nearest` | An average of class 11 (water) and 41 (forest) is nonsense |

ESRI equivalents: **Project Raster** (reproject), **Resample**, **Extract by Mask** (clip).

In [ ]:
# [Python] Reproject with rasterio.warp — the misaligned DEM onto the course grid
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling

src_path, dst_path = "data/mod6/dem_4326.tif", "data/mod6/dem_warped_32610.tif"

with rasterio.open(src_path) as src:
    # 1. Compute the output grid: CRS + transform + dimensions
    dst_transform, dst_w, dst_h = calculate_default_transform(
        src.crs, "EPSG:32610", src.width, src.height, *src.bounds,
        resolution=30.0)                                  # force the course cell size

    profile = src.profile | dict(crs="EPSG:32610", transform=dst_transform,
                                 width=dst_w, height=dst_h)

    # 2. Reproject band-by-band — bilinear because elevation is continuous
    with rasterio.open(dst_path, "w", **profile) as dst:
        reproject(source=rasterio.band(src, 1),
                  destination=rasterio.band(dst, 1),
                  resampling=Resampling.bilinear)

inspect_raster(dst_path)      # ✔ CRS and resolution now match the land cover grid

In [ ]:
# [Python] Resample and clip — coarsen for regional overview, clip to one watershed
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.mask import mask as rio_mask
from rasterio.transform import from_origin
import geopandas as gpd

# ── Resample: 30 m → 90 m (3x coarser) for a fast regional overview ──────────
with rasterio.open("data/mod6/dem_32610.tif") as src:
    scale = 3
    coarse = src.read(1, masked=True,
                      out_shape=(src.height // scale, src.width // scale),
                      resampling=Resampling.average)      # average is fine: continuous
    print(f"Resampled {src.shape} → {coarse.shape} "
          f"(cell {src.res[0]:.0f} m → {src.res[0]*scale:.0f} m)")

# ⚠ For land cover, the SAME operation must use Resampling.mode or .nearest

# ── Clip: Extract by Mask, scripted ──────────────────────────────────────────
watersheds = gpd.read_file("data/mod6/watersheds.gpkg")
cedar = watersheds.loc[watersheds.ws_name == "Cedar Creek"]

with rasterio.open("data/mod6/dem_32610.tif") as src:
    clipped, clip_transform = rio_mask(src, cedar.geometry, crop=True, nodata=src.nodata)
    clip_profile = src.profile | dict(height=clipped.shape[1], width=clipped.shape[2],
                                      transform=clip_transform)

with rasterio.open("data/mod6/dem_cedar.tif", "w", **clip_profile) as dst:
    dst.write(clipped)

inspect_raster("data/mod6/dem_cedar.tif")

In [ ]:
# [R] The same three warps with terra: project(), resample(), crop() + mask()
library(terra)

dem_4326 <- rast("data/mod6/dem_4326.tif")
dem      <- rast("data/mod6/dem_32610.tif")
lc       <- rast("data/mod6/landcover_32610.tif")

# ── Reproject: bilinear for the continuous DEM ───────────────────────────────
dem_warped <- project(dem_4326, dem, method = "bilinear")   # target grid from `dem`
dem_warped

# ── Resample onto another raster's grid — method follows data type ───────────
dem_coarse <- aggregate(dem, fact = 3, fun = "mean")        # 30 m → 90 m
lc_on_coarse <- resample(lc, dem_coarse, method = "near")   # categorical → near!

# ── Clip = crop (extent) + mask (shape) ──────────────────────────────────────
library(sf)
ws    <- st_read("data/mod6/watersheds.gpkg", quiet = TRUE)
cedar <- vect(ws[ws$ws_name == "Cedar Creek", ])

dem_cedar <- mask(crop(dem, cedar), cedar)
plot(dem_cedar, main = "Cedar Creek DEM (crop + mask)")

# ESRI equivalents: Project Raster · Resample · Extract by Mask

> 🔧 **Try it yourself:** Re-run the reproject cell but change `Resampling.bilinear` to `Resampling.nearest`, then compare `inspect_raster()` value ranges. The differences are small for a smooth DEM — now explain why the *reverse* mistake (bilinear on land cover) is catastrophic rather than small.

---
## Section 4 — Neighborhood Operations and Terrain Derivatives

**Focal** operations compute each output cell from a *moving window* of its neighbors — a 3×3 mean smooths noise; a 3×3 range highlights edges. The most-used focal operations in physical geography are the **terrain derivatives**: slope, aspect, and hillshade are all functions of a cell's 3×3 neighborhood on a DEM.

Salmon habitat connection: stream reaches flanked by **steep slopes** deliver more sediment; **low-slope, low-elevation** cells near water are candidate floodplain and off-channel habitat. Slope is the second column of our board request.

ESRI equivalents: **Focal Statistics**, **Slope**, **Aspect**, **Hillshade** (Spatial Analyst).

In [ ]:
# [Python] Focal statistics with SciPy — a moving window over the array
import numpy as np
import rasterio
from scipy import ndimage

with rasterio.open("data/mod6/dem_32610.tif") as src:
    dem = src.read(1)
    cell = src.res[0]
    dem_profile = src.profile

# ── Focal mean (3x3): smooths micro-relief before derivative calculations ────
dem_smooth = ndimage.uniform_filter(dem, size=3)

# ── Focal range (3x3): a quick local-ruggedness measure ──────────────────────
f_max = ndimage.maximum_filter(dem, size=3)
f_min = ndimage.minimum_filter(dem, size=3)
ruggedness = f_max - f_min

print(f"Mean 3x3 local relief: {ruggedness.mean():.1f} m "
      f"(max {ruggedness.max():.1f} m)")

In [ ]:
# [Python] Terrain derivatives — slope and aspect from first principles
import numpy as np
import rasterio
import matplotlib.pyplot as plt

# np.gradient returns dz/dy, dz/dx using each cell's neighbors — a focal operation
dzdy, dzdx = np.gradient(dem_smooth, cell)          # `cell` = 30 m from previous cell

slope_deg  = np.degrees(np.arctan(np.hypot(dzdx, dzdy)))
aspect_deg = (np.degrees(np.arctan2(-dzdx, dzdy)) + 360) % 360   # 0° = north, clockwise

with rasterio.open("data/mod6/slope.tif", "w", **dem_profile) as dst:
    dst.write(slope_deg.astype("float32"), 1)

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
im0 = ax[0].imshow(slope_deg, cmap="magma"); ax[0].set_title("Slope (°)")
plt.colorbar(im0, ax=ax[0], shrink=0.8)
im1 = ax[1].imshow(aspect_deg, cmap="twilight"); ax[1].set_title("Aspect (°)")
plt.colorbar(im1, ax=ax[1], shrink=0.8)
plt.tight_layout(); plt.show()

print(f"Slope: mean {slope_deg.mean():.1f}°, max {slope_deg.max():.1f}°")
print(f"Cells steeper than 20°: {(slope_deg > 20).sum()} — sediment-delivery flags")

In [ ]:
# [R] focal() and terrain() with terra — one-liners for the same operations
library(terra)

dem <- rast("data/mod6/dem_32610.tif")

# ── Focal statistics: w = the moving window ──────────────────────────────────
dem_smooth <- focal(dem, w = 3, fun = "mean")
ruggedness <- focal(dem, w = 3, fun = "max") - focal(dem, w = 3, fun = "min")

# ── Terrain derivatives: slope, aspect (and TRI, TPI, roughness, flowdir) ────
terr <- terrain(dem_smooth, v = c("slope", "aspect"), unit = "degrees")
terr

writeRaster(terr$slope, "data/mod6/slope_r.tif", overwrite = TRUE)

par(mfrow = c(1, 2))
plot(terr$slope,  main = "Slope (°)")
plot(terr$aspect, main = "Aspect (°)")

# Bonus: hillshade for cartography — shade(slope_rad, aspect_rad) wants radians
terr_rad <- terrain(dem_smooth, v = c("slope", "aspect"), unit = "radians")
hs <- shade(terr_rad$slope, terr_rad$aspect, angle = 45, direction = 315)

> 🔧 **Try it yourself:** Increase the focal window from 3×3 to 9×9 in either language and recompute slope. Does mean slope go up or down? Why does window size matter when you report "mean watershed slope" to the recovery board?

---
## Section 5 — Zonal Statistics: Summarizing Rasters by Vector Zones

This is the module's payoff and the heart of the board's request: *collapse a raster into one row per polygon.* Mean elevation **by watershed**, percent forest **by watershed** — a continuous surface becomes a table you can join, rank, and report.

Zonal statistics is also where the vector world (Modules 3–4) and the raster world (Modules 5–6) finally meet: the **zones** are polygons, the **values** are a raster.

Two paths, per course convention — **ESRI first**, then open-source. Both produce the same table.

| Path | Tools | Runs on |
|---|---|---|
| ESRI | `arcpy.sa.ZonalStatisticsAsTable`, R-ArcGIS Bridge (`arcgisbinding`) | 🏠 Local only — Pro + Spatial Analyst |
| Open-source | `rasterstats` (Python), `exactextractr` / `terra::extract` (R) | JupyterHub, anywhere |

In [ ]:
# [Python / ArcPy] ESRI path — the Zonal Statistics as Table tool, scripted
# Reference cell: requires ArcGIS Pro + Spatial Analyst. Falls back on JupyterHub.

try:
    import arcpy
    from arcpy.sa import ZonalStatisticsAsTable

    arcpy.CheckOutExtension("Spatial")
    arcpy.env.workspace = "data/mod6"
    arcpy.env.overwriteOutput = True

    # Zone field must be integer or text — ws_id qualifies
    ZonalStatisticsAsTable(in_zone_data="watersheds.gpkg/watersheds",
                           zone_field="ws_id",
                           in_value_raster="dem_32610.tif",
                           out_table="zonal_elev.dbf",
                           statistics_type="ALL")

    # Join the summary back onto the polygons — one row per watershed
    arcpy.management.JoinField("watersheds.gpkg/watersheds", "ws_id",
                               "zonal_elev.dbf", "ws_id",
                               ["MEAN", "MIN", "MAX", "STD"])
    arcpy.CheckInExtension("Spatial")
    print("Zonal statistics joined to watersheds.")

except ModuleNotFoundError:
    print("arcpy not available (expected on JupyterHub) — reference cell only.")

In [ ]:
# [R] LOCAL ONLY — Run in RStudio or Positron
# ESRI path via the R-ArcGIS Bridge: the Bridge MOVES data, it doesn't run tools.
# Pattern: read Pro data into R → compute with R strengths → arc.write() back.

# library(arcgisbinding)
# library(exactextractr)
# library(sf)
# library(terra)
#
# arc.check_product()                                   # verifies the Pro license
#
# ws_arc <- arc.open("C:/gis/habitat.gdb/watersheds")   # feature class in a gdb
# ws     <- arc.data2sf(arc.select(ws_arc))             # → sf object
# dem    <- rast("C:/gis/rasters/dem_32610.tif")
#
# ws$elev_mean <- exact_extract(dem, ws, "mean")        # zonal, R-side
#
# arc.write("C:/gis/habitat.gdb/ws_zonal", ws,          # back into the gdb —
#           overwrite = TRUE)                           # Pro sees it immediately

cat("Reference cell — requires ArcGIS Pro + arcgisbinding on your machine.\n")

In [ ]:
# [Python] Open-source path — rasterstats: zero licensing, runs anywhere
# 📋 [Terminal] if needed:  pip install rasterstats
import geopandas as gpd
import pandas as pd
from rasterstats import zonal_stats

watersheds = gpd.read_file("data/mod6/watersheds.gpkg")

# ── Continuous raster: mean/min/max elevation and mean slope per watershed ───
elev = pd.DataFrame(zonal_stats(watersheds, "data/mod6/dem_32610.tif",
                                stats=["mean", "min", "max", "std"]))
slope = pd.DataFrame(zonal_stats(watersheds, "data/mod6/slope.tif",
                                 stats=["mean"]))

watersheds["elev_mean"]  = elev["mean"].round(1)
watersheds["elev_range"] = (elev["max"] - elev["min"]).round(1)
watersheds["slope_mean"] = slope["mean"].round(1)

# ── Categorical raster: land cover composition per watershed ─────────────────
lc_counts = zonal_stats(watersheds, "data/mod6/landcover_32610.tif",
                        categorical=True)                 # dict of {class: count}
FOREST = {41, 42}
watersheds["pct_forest"] = [
    round(100 * sum(c for k, c in d.items() if k in FOREST) / sum(d.values()), 1)
    for d in lc_counts]

print(watersheds[["ws_name", "elev_mean", "elev_range", "slope_mean", "pct_forest"]]
      .sort_values("pct_forest", ascending=False).to_string(index=False))

In [ ]:
# [R] Open-source path — terra::extract and exactextractr
library(terra)
library(sf)

dem   <- rast("data/mod6/dem_32610.tif")
slope <- rast("data/mod6/slope_r.tif")
lc    <- rast("data/mod6/landcover_32610.tif")
ws    <- st_read("data/mod6/watersheds.gpkg", quiet = TRUE)
ws_v  <- vect(ws)

# ── terra::extract — zonal summary per polygon ───────────────────────────────
ws$elev_mean  <- extract(dem,   ws_v, fun = mean, na.rm = TRUE)[, 2]
ws$slope_mean <- extract(slope, ws_v, fun = mean, na.rm = TRUE)[, 2]

# ── exactextractr — faster, weights partial edge cells correctly ─────────────
# library(exactextractr)
# ws$elev_mean <- exact_extract(dem, ws, "mean")

# ── Categorical composition: cross-tabulate zones x classes ──────────────────
ws_rast <- rasterize(ws_v, lc, field = "ws_id")   # preview of Section 6!
comp <- crosstab(c(ws_rast, lc))
head(comp)

print(st_drop_geometry(ws[, c("ws_name", "elev_mean", "slope_mean")]))

> 🔧 **Try it yourself:** Add `"median"` and `"count"` to the elevation `zonal_stats()` call. For which watershed do mean and median elevation differ most — and what does that tell you about its terrain distribution? Verify by clipping and plotting that watershed's histogram (Module 5 skills).

---
## Section 6 — Rasterize and Vectorize: Crossing the Divide

Sometimes the analysis demands the *other* data model:

- **Rasterize (vector → raster):** burn polygon attributes into a grid — zonal logic at grid speed, masks for map algebra, inputs for models that only eat arrays.
- **Vectorize (raster → vector):** trace class boundaries into polygons — so results can join attribute workflows, feed web maps, or land in a geodatabase.

**Rule:** vectorize **categorical** rasters only. Polygonizing a continuous DEM produces one polygon per cell — reclassify first, then vectorize.

ESRI equivalents: **Polygon to Raster**, **Raster to Polygon** (Conversion toolbox).

In [ ]:
# [Python] rasterio.features — rasterize zones, vectorize classes
import rasterio
from rasterio import features
import geopandas as gpd
import numpy as np
from shapely.geometry import shape

watersheds = gpd.read_file("data/mod6/watersheds.gpkg")

with rasterio.open("data/mod6/dem_32610.tif") as src:
    out_shape, transform = src.shape, src.transform
    base_profile = src.profile

# ── Vector → raster: burn ws_id into a grid aligned with the DEM ─────────────
ws_grid = features.rasterize(
    zip(watersheds.geometry, watersheds.ws_id),
    out_shape=out_shape, transform=transform,
    fill=0, dtype="uint16")

with rasterio.open("data/mod6/ws_grid.tif", "w",
                   **(base_profile | {"dtype": "uint16", "nodata": 0})) as dst:
    dst.write(ws_grid, 1)
print(f"Rasterized watershed IDs: {sorted(int(v) for v in np.unique(ws_grid))}")

# ── Raster → vector: polygonize the Section 2 elevation classes ──────────────
with rasterio.open("data/mod6/elev_class.tif") as src:
    elev_class = src.read(1)

polys = [{"geometry": shape(g), "elev_class": int(v)}
         for g, v in features.shapes(elev_class, mask=elev_class > 0,
                                     transform=transform)]
class_gdf = (gpd.GeoDataFrame(polys, crs="EPSG:32610")
               .dissolve(by="elev_class", as_index=False))
class_gdf["area_km2"] = (class_gdf.area / 1e6).round(2)
print(class_gdf[["elev_class", "area_km2"]].to_string(index=False))

class_gdf.to_file("data/mod6/elev_class_polys.gpkg", layer="elev_classes")

In [ ]:
# [R] terra: rasterize() and as.polygons()
library(terra)
library(sf)

dem <- rast("data/mod6/dem_32610.tif")
ws  <- vect("data/mod6/watersheds.gpkg")

# ── Vector → raster ──────────────────────────────────────────────────────────
ws_grid <- rasterize(ws, dem, field = "ws_id")
plot(ws_grid, main = "Watershed IDs as a grid")

# ── Raster → vector: class polygons back to sf for attribute workflows ───────
elev_class <- rast("data/mod6/elev_class_r.tif")
class_polys <- as.polygons(elev_class)          # dissolves by value automatically
class_sf    <- st_as_sf(class_polys)
class_sf$area_km2 <- round(as.numeric(st_area(class_sf)) / 1e6, 2)
print(st_drop_geometry(class_sf))

st_write(class_sf, "data/mod6/elev_class_polys_r.gpkg", delete_dsn = TRUE, quiet = TRUE)

# ESRI equivalents: arcpy.conversion.PolygonToRaster / RasterToPolygon

> 🔧 **Try it yourself:** Rasterize the watersheds again, but burn `ws_id` onto the **land cover** grid instead of the DEM grid. Same code, different template raster. Then confirm with `inspect_raster()` that the output aligns with `landcover_32610.tif` — alignment comes from the template, not from the vector data.

---
## Section 7 — Multidimensional Arrays: Time Series with xarray and stars

Scientific datasets — climate reanalysis, satellite time series, ocean models — arrive as **NetCDF** or **HDF** files: rasters with *extra dimensions*. Our NDVI file is a 3-D cube: `time × y × x`. A GeoTIFF-only toolchain would force you to juggle 12 separate files; `xarray` (Python) and `stars` (R) treat the cube as one labeled object.

Two ideas do most of the work:

- **Named dimensions and coordinates** — you write `sel(time="2025-07")`, not `arr[6, :, :]`.
- **Reduction across a dimension** — `.mean(dim="time")` collapses 12 grids into one per-pixel average; `.mean(dim=["y", "x"])` collapses each grid into one number per month.

ESRI equivalent: multidimensional raster tools / **Aggregate Multidimensional Raster** on a CRF.

In [ ]:
# [Python] xarray — open the cube, slice time, reduce across dimensions
import xarray as xr
import matplotlib.pyplot as plt

ds = xr.open_dataset("data/mod6/ndvi_monthly.nc")
print(ds)                                        # dims, coords, variables — always look first

ndvi = ds["ndvi"]

# ── Temporal slice: label-based selection, no index arithmetic ────────────────
july = ndvi.sel(time="2025-07-15")

# ── Reduce across time: per-pixel statistics over the year ────────────────────
ndvi_max   = ndvi.max(dim="time")                # peak greenness per pixel
ndvi_range = ndvi.max(dim="time") - ndvi.min(dim="time")   # seasonal amplitude

# ── Reduce across space: the regional phenology curve ────────────────────────
monthly_mean = ndvi.mean(dim=["y", "x"])

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
july.plot(ax=ax[0], cmap="YlGn", vmin=0, vmax=0.9); ax[0].set_title("NDVI — July")
ndvi_range.plot(ax=ax[1], cmap="viridis");          ax[1].set_title("Seasonal amplitude")
monthly_mean.plot(ax=ax[2], marker="o");            ax[2].set_title("Study-area mean by month")
plt.tight_layout(); plt.show()

peak = monthly_mean.idxmax(dim="time").values
print(f"Peak regional greenness: {str(peak)[:10]}")

In [ ]:
# [R] stars — the same cube with dimension-aware R
library(stars)

nc <- read_ncdf("data/mod6/ndvi_monthly.nc")
nc                                          # note the time / y / x dimensions

# ── Temporal slice ────────────────────────────────────────────────────────────
july <- nc |> dplyr::slice(time, 7)
plot(july, main = "NDVI — July")

# ── Reduce across time: st_apply over the spatial dims keeps x,y; drops time ──
ndvi_mean  <- st_apply(nc, c("x", "y"), mean)
ndvi_range <- st_apply(nc, c("x", "y"), function(v) max(v) - min(v))
plot(ndvi_range, main = "Seasonal amplitude")

# ── Reduce across space: one value per month → the phenology curve ────────────
monthly <- st_apply(nc, "time", mean, na.rm = TRUE)
plot(seq_len(12), monthly[[1]], type = "b", xlab = "month", ylab = "mean NDVI")

> 🔧 **Try it yourself:** Compute the **growing-season mean** (May–September) per pixel: in xarray, `ndvi.sel(time=slice("2025-05", "2025-09")).mean(dim="time")`. Then subtract the annual mean from it. Where is the difference largest, and how does that pattern relate to the land cover grid?

---
## Section 8 — Guided Lab: The Habitat Prioritization Request

**Scenario, closed:** the recovery board's table is due. For **every watershed**: mean elevation, mean slope, percent forest, and a floodplain flag (any cells below 100 m). You now own every step:

| Step | Operation | Section |
|---|---|---|
| 1 | Inspect all inputs; warp the delivered DEM onto the course grid (EPSG:32610, 30 m) | 1, 3 |
| 2 | Derive slope from the warped DEM | 4 |
| 3 | Map algebra: build the below-100 m floodplain mask | 2 |
| 4 | Zonal statistics: elevation, slope, forest %, floodplain count per watershed | 5 |
| 5 | Join results to the polygons, rank by habitat score, export a GeoPackage | 5, M3 |

Work through the cell below. It is complete through step 4 — **step 5's ranking is yours to finish** at the `# 🔧 TODO` markers.

In [ ]:
# [Python] Lab — The Habitat Prioritization Request
import numpy as np
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
import geopandas as gpd
import pandas as pd
from rasterstats import zonal_stats
from scipy import ndimage

# ── Step 1: inspect, then warp the delivered DEM onto the course grid ────────
inspect_raster("data/mod6/dem_4326.tif")

with rasterio.open("data/mod6/dem_4326.tif") as src:
    t, w, h = calculate_default_transform(src.crs, "EPSG:32610",
                                          src.width, src.height, *src.bounds,
                                          resolution=30.0)
    profile = src.profile | dict(crs="EPSG:32610", transform=t, width=w, height=h)
    with rasterio.open("data/mod6/lab_dem.tif", "w", **profile) as dst:
        reproject(rasterio.band(src, 1), rasterio.band(dst, 1),
                  resampling=Resampling.bilinear)

# ── Step 2: slope from the warped DEM ────────────────────────────────────────
with rasterio.open("data/mod6/lab_dem.tif") as src:
    dem = src.read(1)
    cell = src.res[0]
    lab_profile = src.profile
dzdy, dzdx = np.gradient(ndimage.uniform_filter(dem, 3), cell)
slope = np.degrees(np.arctan(np.hypot(dzdx, dzdy))).astype("float32")
with rasterio.open("data/mod6/lab_slope.tif", "w", **lab_profile) as dst:
    dst.write(slope, 1)

# ── Step 3: floodplain mask via map algebra ──────────────────────────────────
floodplain = (dem < 100).astype("uint8")
with rasterio.open("data/mod6/lab_floodplain.tif", "w",
                   **(lab_profile | {"dtype": "uint8", "nodata": 255})) as dst:
    dst.write(floodplain, 1)

# ── Step 4: zonal statistics per watershed ───────────────────────────────────
ws = gpd.read_file("data/mod6/watersheds.gpkg")

ws["elev_mean"]  = pd.DataFrame(zonal_stats(ws, "data/mod6/lab_dem.tif",
                                            stats=["mean"]))["mean"].round(1)
ws["slope_mean"] = pd.DataFrame(zonal_stats(ws, "data/mod6/lab_slope.tif",
                                            stats=["mean"]))["mean"].round(1)
ws["flood_cells"] = pd.DataFrame(zonal_stats(ws, "data/mod6/lab_floodplain.tif",
                                             stats=["sum"]))["sum"].astype(int)

lc_counts = zonal_stats(ws, "data/mod6/landcover_32610.tif", categorical=True)
ws["pct_forest"] = [round(100 * sum(c for k, c in d.items() if k in {41, 42})
                          / sum(d.values()), 1) for d in lc_counts]

# ── Step 5: rank and deliver ─────────────────────────────────────────────────
ws["has_floodplain"] = ws["flood_cells"] > 0

# 🔧 TODO (a): build a simple habitat_score — e.g. pct_forest weighted 0.5,
#              a floodplain bonus of 20 points, minus slope_mean weighted 0.5.
# ws["habitat_score"] = ...

# 🔧 TODO (b): sort by your score, print the ranked table, and export:
# ws.to_file("data/mod6/habitat_priority.gpkg", layer="watersheds")

print(ws[["ws_name", "elev_mean", "slope_mean", "pct_forest",
          "flood_cells", "has_floodplain"]].to_string(index=False))

In [ ]:
# [R] Lab — The Habitat Prioritization Request (R version)
library(terra)
library(sf)

# ── Step 1: warp the delivered DEM onto the course grid ──────────────────────
target <- rast("data/mod6/dem_32610.tif")          # grid template: CRS + res + extent
dem    <- project(rast("data/mod6/dem_4326.tif"), target, method = "bilinear")

# ── Step 2: slope ─────────────────────────────────────────────────────────────
slope <- terrain(focal(dem, w = 3, fun = "mean"), v = "slope", unit = "degrees")

# ── Step 3: floodplain mask ──────────────────────────────────────────────────
floodplain <- dem < 100

# ── Step 4: zonal statistics ─────────────────────────────────────────────────
ws   <- st_read("data/mod6/watersheds.gpkg", quiet = TRUE)
ws_v <- vect(ws)

ws$elev_mean   <- extract(dem,        ws_v, fun = mean, na.rm = TRUE)[, 2] |> round(1)
ws$slope_mean  <- extract(slope,      ws_v, fun = mean, na.rm = TRUE)[, 2] |> round(1)
ws$flood_cells <- extract(floodplain, ws_v, fun = sum,  na.rm = TRUE)[, 2]

lc <- rast("data/mod6/landcover_32610.tif")
pct_forest <- function(v, ...) round(100 * mean(v %in% c(41, 42), na.rm = TRUE), 1)
ws$pct_forest <- extract(lc, ws_v, fun = pct_forest)[, 2]

# ── Step 5: rank and deliver ─────────────────────────────────────────────────
ws$has_floodplain <- ws$flood_cells > 0

# 🔧 TODO (a): ws$habitat_score <- ...
# 🔧 TODO (b): st_write(ws, "data/mod6/habitat_priority_r.gpkg", delete_dsn = TRUE)

print(st_drop_geometry(ws[, c("ws_name", "elev_mean", "slope_mean",
                              "pct_forest", "has_floodplain")]))

---
## Section 9 — Extended Application: `raster_pipeline()`

**Scenario extension:** next quarter the board sends a new DEM, a new land cover grid, and a revised watershed layer — and wants the same table. Parameterize the lab into a function, exactly as `build_permit_deliverable()` did for vectors in Module 3. New data, zero code changes.

This function joins `inspect_raster()` (Module 5) in your personal toolkit and carries forward into the full workflow pipelines of Modules 9–11.

In [ ]:
# [Python] A reusable, parameterized raster workflow function
import numpy as np
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
import geopandas as gpd
import pandas as pd
from rasterstats import zonal_stats
from scipy import ndimage
from pathlib import Path

def raster_pipeline(dem_path, landcover_path, zones_path, out_gpkg,
                    target_crs="EPSG:32610", resolution=30.0,
                    forest_classes={41, 42}, floodplain_below=100.0):
    """Warp a DEM to the target grid, derive slope and a floodplain mask,
    summarize everything by zone polygons, and export a GeoPackage.

    Returns the zones GeoDataFrame with summary columns attached."""
    out_gpkg = Path(out_gpkg)
    work = out_gpkg.parent
    work.mkdir(parents=True, exist_ok=True)

    # 1 ── Warp the DEM onto the target grid (bilinear: continuous data)
    warped = work / "_pipe_dem.tif"
    with rasterio.open(dem_path) as src:
        if src.crs.to_string() != target_crs or abs(src.res[0] - resolution) > 1e-6:
            t, w, h = calculate_default_transform(
                src.crs, target_crs, src.width, src.height, *src.bounds,
                resolution=resolution)
            profile = src.profile | dict(crs=target_crs, transform=t, width=w, height=h)
            with rasterio.open(warped, "w", **profile) as dst:
                reproject(rasterio.band(src, 1), rasterio.band(dst, 1),
                          resampling=Resampling.bilinear)
        else:
            warped = Path(dem_path)                      # already on the grid

    # 2 ── Derivatives: slope + floodplain mask
    with rasterio.open(warped) as src:
        dem, cell, profile = src.read(1), src.res[0], src.profile
    dzdy, dzdx = np.gradient(ndimage.uniform_filter(dem, 3), cell)
    slope = np.degrees(np.arctan(np.hypot(dzdx, dzdy))).astype("float32")
    flood = (dem < floodplain_below).astype("uint8")

    slope_tif, flood_tif = work / "_pipe_slope.tif", work / "_pipe_flood.tif"
    with rasterio.open(slope_tif, "w", **profile) as dst:
        dst.write(slope, 1)
    with rasterio.open(flood_tif, "w",
                       **(profile | {"dtype": "uint8", "nodata": 255})) as dst:
        dst.write(flood, 1)

    # 3 ── Zonal statistics — reproject zones defensively first (Module 3 habit)
    zones = gpd.read_file(zones_path).to_crs(target_crs)
    zones["elev_mean"]  = pd.DataFrame(zonal_stats(zones, warped,
                                       stats=["mean"]))["mean"].round(1)
    zones["slope_mean"] = pd.DataFrame(zonal_stats(zones, slope_tif,
                                       stats=["mean"]))["mean"].round(1)
    zones["flood_cells"] = pd.DataFrame(zonal_stats(zones, flood_tif,
                                        stats=["sum"]))["sum"].fillna(0).astype(int)
    cats = zonal_stats(zones, landcover_path, categorical=True)
    zones["pct_forest"] = [
        round(100 * sum(c for k, c in d.items() if k in forest_classes)
              / sum(d.values()), 1) if d else np.nan for d in cats]

    # 4 ── Deliver: multi-layer GeoPackage (Module 3 output standard)
    zones.to_file(out_gpkg, layer="zone_summary", driver="GPKG")
    print(f"✔ {len(zones)} zones summarized → {out_gpkg}")
    return zones

# One call replaces the Pro click-path — and reruns on any new delivery:
result = raster_pipeline(dem_path="data/mod6/dem_4326.tif",
                         landcover_path="data/mod6/landcover_32610.tif",
                         zones_path="data/mod6/watersheds.gpkg",
                         out_gpkg="data/mod6/habitat_deliverable.gpkg")
result[["ws_name", "elev_mean", "slope_mean", "pct_forest", "flood_cells"]]

> 🔧 **Try it yourself:** Call `raster_pipeline()` again with `floodplain_below=80` and `forest_classes={42}` (evergreen only). Which watershed's ranking changes? A parameterized function turns a "what if the threshold were…" board question into a 10-second answer.

---

## 🔍 Module 6 — Self-Check Questions

Work through these without looking at the cells above. Then verify by running code.

**Map algebra and reclassification**
1. In Tomlin's taxonomy, what distinguishes a *local* operation from a *focal* one? Give one example of each from this notebook.
2. You need every DEM cell converted from metres to feet in ArcPy. Write the one-line `arcpy.sa` expression.
3. In R, what three columns does the matrix passed to `terra::classify()` contain?

**Warping**
4. A colleague hands you a land cover raster in EPSG:4326 and you reproject it with bilinear resampling. What specifically goes wrong, and which method should you have used?
5. What three grid properties must two rasters share before you can combine them in map algebra?
6. Why is "warp once, early" better practice than reprojecting on the fly inside every operation?

**Focal operations and terrain**
7. Slope, aspect, and hillshade are all *focal* operations. What is the neighborhood, and what happens to the output at the raster's edge?
8. In Python, `np.gradient(dem, cell)` needs the cell size argument. What units does slope come out in if you forget it, and why is the result wrong rather than just unscaled?

**Zonal statistics and rasterize/vectorize**
9. `zonal_stats()` with `categorical=True` returns something different from `stats=["mean"]`. What, and why is `mean` meaningless for land cover?
10. The R-ArcGIS Bridge "moves data, it doesn't run tools." Explain what that means for how you compute zonal statistics on the ESRI path in R.
11. Why should you never run Raster to Polygon on a raw DEM? What one operation makes it safe?

**Multidimensional arrays**
12. In xarray, what's the difference between `ndvi.mean(dim="time")` and `ndvi.mean(dim=["y", "x"])` — what shape does each return, and what question does each answer?

---
## ✅ Module 6 Homework — Deliverables

Submit the following in Canvas before the next session:

### Deliverable 1 — Warp and Document (25 pts)
Take one raster that does **not** match the course grid (use your own agency data, or `dem_4326.tif` from this notebook):
- Run `inspect_raster()` (or the terra equivalent) **before and after** warping
- Reproject to EPSG:32610 at 30 m, choosing and **justifying** your resampling method in one sentence
- Clip the result to a polygon boundary

Submit: the before/after inspection output and your resampling justification.

### Deliverable 2 — Zonal Report (25 pts)
Using any polygon zones (your own, or the watersheds from this notebook):
- Compute zonal statistics from **one continuous raster** (at least two statistics) **and one categorical raster** (class composition)
- Join all results back onto the polygons
- Export the joined layer as a GeoPackage and symbolize one summary column in QGIS or ArcGIS Pro

Submit: a screenshot of the summary table **and** a screenshot of the symbolized map.

### Deliverable 3 — Parameterized Raster Pipeline (50 pts)
Adapt `raster_pipeline()` (or write your own) to a real or realistic raster workflow:
- The function must take at least **three** file paths as arguments (a value raster, a categorical raster, and a zones layer)
- It must warp inputs onto a shared grid, deriving at least **one** map-algebra or terrain product along the way
- It must produce a zonal summary joined to the zones
- It must export a GeoPackage
- Commit the notebook and output GeoPackage to your GitHub repo

Submit: your GitHub repo link (same repo from Modules 2–5). Tag the commit `module6-deliverable`.

---

**Stuck?** Post in Ed Discussion — tag `#module6`. Include your error message, the cell that failed, and your OS/environment. If you solved something tricky, share how — the whole cohort benefits.